In [277]:
%run CertificateClassGroupLean2.0.ipynb
load('IrreducibilityLeanProofWriter.sage')

In [278]:
R.<X>=PolynomialRing(ZZ)

In [279]:
K.<a> = NumberField(X^5 + 20*X - 16)

In [280]:
B = [1, a, 1/2*a^2, 1/4*a^3 - 1/2*a, 1/4*a^4]

In [281]:
O = K.ring_of_integers()

In [223]:
F = factor(O.ideal(3))

In [230]:
list(F[1][0].gens())

[3, -1/8*a^4 + 3/8*a^3 - 11/8*a^2 + 35/8*a - 1/4]

In [166]:
#ideal_gens = [3, -5/24*a^4 + 1/8*a^3 - 47/24*a^2 + 59/24*a + 13/4]

In [282]:
#Computes a primitive element for the field O/I. 

def RandomPrimitiveElementQuot(K,ideal_gens,p):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    a = K.gen()
    flag = 0 
    cont = 0 
    while flag == 0 and cont < 100:
        g = O.random_element()
        cont = cont + 1
        poly = g.minpoly()
        F = factor(GF(p)[x](g.minpoly()))
        for f in F:
            if p ** f[0].degree() == I.norm() and ((ZZ[X](f[0])(g)) in I):
                flag = 1
                return g, f[0]

In [283]:
#Writes a proof of primality for the ideal given by its generators (as an O-module). The name of the ideal 
# is index by the prime above it and the label. So ideal_namepNlabel. 

def PrimalityCertificate (K, B, ideal_gens, p, ideal_name, label, name_prime_proof):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    N = I.norm()
    ideal_name_full = ideal_name + str(p) + 'N' + str(label)
    gensc = [elems_to_basis([x], B).list() for x in ideal_gens]
    print(f'def {ideal_name_full} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    print('')
    span_label = 'SI'+str(p)+'N'+str(label)
    w = IdealEqSpanCertificateLean(K, ideal_gens, B, span_label, ideal_name_full)
    ik, jk = FindNzEntry(w) 
    print('')
    print(f'lemma N{ideal_name_full} : Nat.card (O ⧸ {ideal_name_full}) = {N} := ')
    print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label})")            
    print('')
    
    if N == p: 
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := prime_ideal_of_norm_prime {name_prime_proof} _ N{ideal_name_full}')
    else: 
        g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
        a = list((elems_to_basis([g],B)).transpose()[0])
        c = list((elems_to_basis([ZZ[X](f)(g)],B)).transpose()[0])
        IdealMemZLean(K, B,ideal_gens, c, 'MemC'+ideal_name_full, ideal_name_full, span_label)
        print('')
        CertificateIrrModpFFP(GF(p)[X](f),p,label,sys.stdout)
        print(f'''
def P{ideal_name_full} : PrimeIdeal O {p} where 
  r := {len(B)}
  n := {f.degree()}
  hpos := by decide
  TT := timesTableO
  T := Table
  heq := timesTableT_eq_Table
  I := {ideal_name_full}
  hcard := N{ideal_name_full}
  P := {list(f)}
  f := ofList {list(f)}
  hfeq := by decide
  hirr := irreducible_ofList_ofCertificateIrreducibleZModOfList' P{p}P{label}
  hneq := by decide
  hlen := by decide
  c := !{c}
  a := !{a}
  z := ![1,0,0,0,0]
  hBz := B_one_repr
  hpol := by decide
  hcmem := mem_of_certificate O ℤ _ _ _ _ MemC{ideal_name_full}
  hpmem := by 
    show  _ ∈ {ideal_name_full}.carrier
    rw [ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label}]
    apply Submodule.subset_span
    use 0 ; dsimp ; congr ; norm_num
        
        ''')
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := PrimeIdeal_isPrime P{ideal_name_full}')
        
        

    
   
   # g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
    

In [284]:
def PrimesBelowGens(K,p):
    O = K.ring_of_integers() 
    F = factor(O.ideal(p))
    ideal_gens = [[[], F[i][1]] for i in range(len(F))]
    for i in range(len(F)):
        ideal_gens[i][0] = list(F[i][0].gens_reduced())
    return ideal_gens
    

In [302]:
p = 61
F = PrimesBelowGens(K,p)
l = len(F)

for i in range(l):
    if l > 1 : 
        PrimalityCertificate(K, B, F[i][0], p, 'I',i,'hp' + str(p) + '.out')
    else: 
        PrimalityCertificate(K, B, F[i][0], p, 'I','','hp' + str(p) + '.out')
    
ideal_gens = [F[i][0] for i in range(l)]
exp = [F[i][1] for i in range(l)]

IteratedMulLean(K, B, ideal_gens, exp, 'rfl', 'rfl', 'I' + str(p) + 'N')

ideal_names_list = []

for i in range(l):
    if l > 1 :
        ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N' + str(i)]
    else :
        ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N']

name_ideals_rep = "[" + ", ".join(ideal_names_list) + "]"



print(f'''def PBC{p} : PrimesBelowPCertificate {p} !{name_ideals_rep} where 
  Ip := by 
    intro i 
    fin_cases i ''')
for i in range(len(ideal_names_list)): 
    print('    exact isPrime' + ideal_names_list[i])

if len(ideal_names_list) >= 2 : 
    print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    rw [ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulI{p}N{len(ideal_names_list)-2}, Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
else : 
    print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    unfold I{p}N
    rw [Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
    
  


def I61N : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![61, 0, 0, 0, 0]] i)))

def SI61N: IdealEqSpanCertificate O ℤ timesTableO I61N
 ![![61, 0, 0, 0, 0]] 
 ![![61, 0, 0, 0, 0], ![0, 61, 0, 0, 0], ![0, 0, 61, 0, 0], ![0, 0, 0, 61, 0], ![0, 0, 0, 0, 61]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![61, 0, 0, 0, 0], ![0, 61, 0, 0, 0], ![0, 0, 61, 0, 0], ![0, 0, 0, 61, 0], ![0, 0, 0, 0, 61]]]
  hmulB := by decide
  f := ![![![1, 0, 0, 0, 0]], ![![0, 1, 0, 0, 0]], ![![0, 0, 1, 0, 0]], ![![0, 0, 0, 1, 0]], ![![0, 0, 0, 0, 1]]]
  g := ![![![1, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 1]]]
  hle1 := by decide
  hle2 := by decide

lemma NI61N : Nat.card (O ⧸ I61N) = 844596301 := 
 ideal_norm_eq_prod' B _ _ (by decide) 0 0 (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ SI61N)

def MemCI61N : IdealMemCertificate O ℤ B I61N
  ![![61, 0, 0, 0, 0], ![0, 61, 0, 0, 0], ![0, 0, 61, 0, 0], ![0, 0, 0, 61, 0],

In [490]:
#Relations generator. 

#Given a bound < M, and (minimal) list of generators J for the class group (given in order and in terms of their O-generators), we 
# find the relations between the ideals with norm less than B and the generators of the class group. In case 
# an ideal is principal, no relation is shown (as this corresponds to a trivial one)

#Default J_name is J. 
# name_cert([e0,...,em]) is the the name of a proof of J^e0 * ... * J^em. 

def RelationsGenerator(K, B, M, J_ideal_gens, J_name, name_cert): 
    s = len(J_ideal_gens)
    O = K.ring_of_integers()
    if s > 1 :
        J_name = [J_name + str(i) for i in range(s)]
    else : 
        J_name = [J_name]
    J = [O.ideal(J_ideal_gens[i]) for i in range(s)]
    for p in primes(M): 
        F = PrimesBelowGens(K,p)
        l = len(F)
        for cont in range(l):
            I = O.ideal(F[cont][0])
            if I.norm() < M : 
                # We find (e0, ..., em) such that [I] = [J0]^e0 * ... * [Jm]^em
                expon = [(Cl(I)).exponents()[i] for i in range(s)]
                if expon != [0 for i in range(s)] : 
                    JJ = prod([J[i]**(expon[i]) for i in range(s)])
                    # We find the generator for the principal ideal I/(J^e0 * ... * J^em) 
                    genel = (I/(JJ)).gens_reduced()[0]
                    # We find the denominator of this algebraic number, so that we write it as alpha/d. 
                    d = genel.denominator()
                    for t in prime_divisors(d):
                        while (d % t == 0) and ((d//t) * genel in O):
                            d = d // t
                    x = d * genel

                    alpha = elems_to_basis([x], B).list()

                    # These are generators for alpha * (J^e0 * ... * J^em)

                    A = [x * i for i in (JJ).gens_reduced()]
                    # These are generators for d * I 
                    C = ([d * i for i in I.gens()])
                    CC = I.gens()

                    Gen1 = [elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))]
                    Gen2 = [elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))]

                    before_div = [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))]
                    JJ_gens = [elems_to_basis((JJ).gens_reduced(), B).transpose().rows()[j].list() for j in range(len((JJ).gens_reduced()))]

                    # We have d * I = alpha * (J^e0 * ... * J^em), so we find the combinations between their generators 
                    # that certify this inequality. 

                    h = [IdealLift(K, B, C, A[i]) for i in range(len(A))]
                    g = [IdealLift(K, B, A, C[i]) for i in range(len(C))]

                    # And represent this in terms of our integral basis. 
                    gp = [[elems_to_basis(g[i], B).transpose().rows()[j].list() for j in range(len(A))] for i in range(len(g))]
                    hp = [[elems_to_basis(h[i], B).transpose().rows()[j].list() for j in range(len(C))] for i in range(len(h))]

                    if s < 2: 
                        J_name_exp = J_name[0] + f' ^ {expon[0]}'
                    else : 
                        J_name_exp = J_name[0] + f' ^ {expon[0]}'
                        for i in range(1,s):
                            J_name_exp = J_name_exp + '*' + J_name[i] + f'^ {expon[i]}'

                    print(f'''
def R{p}RS{cont} : IdealMulPrincipalCertificate O ℤ timesTableO ({J_name_exp}) !{alpha} {ExList(str(JJ_gens))}
  {ExList(str(Gen1))} where
    T := Table
    heq := timesTableT_eq_Table
    hI := by
      simp only [pow_succ, pow_one, pow_zero, one_mul]
      {name_cert(expon)}
    hmul := by decide

def E{p}RS{cont} : IdealEqCertificateO O ℤ timesTableO (Ideal.span {{{d}}} * I{p}N{cont}) (Ideal.span {{B.equivFun.symm !{alpha}}} * {J_name_exp}) 
      {ExList(str(Gen2))} {ExList(str(Gen1))} where 
    T := Table
    heq := timesTableT_eq_Table
    hieq1 := by convert ideal_mul_span_singleton_coe O ℤ  _ _ _ rfl {d} ; decide ; exact {{out := by decide}}
    hieq2 := by 
      exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R{p}RS{cont}
    g := {ExList(str(gp))}
    h := {ExList(str(hp))}
    hle1 := by decide
    hle2 := by decide

lemma R{p}N{cont} : Ideal.span {{{d}}} * I{p}N{cont} =  Ideal.span {{B.equivFun.symm !{alpha}}} * {J_name_exp} := by 
  refine ideal_eq_of_IdealEqCertificateO _ _ _ _ _ _ _ E{p}RS{cont}
                    ''')
                
                    
            


In [491]:
def name_cert_example(l) : 
    if l == [1]:
        return 'rfl'
    if l == [2]:
        return 'exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0'
    else :
        return 0
    

In [492]:
Cl = K.class_group('pari')
B = [K(x) for x in B]
J_ideal_gens = [Cl.gens()[0].ideal().gens_reduced()]
RelationsGenerator(K,B,63, J_ideal_gens, 'J', name_cert_example)


def R2RS0 : IdealMulPrincipalCertificate O ℤ timesTableO (J ^ 2) ![0, 1, 0, 0, 0] ![![2, 0, 0, 0, 0], ![1, 0, 1, 0, 0]]
  ![![0, 2, 0, 0, 0], ![0, 2, 0, 2, 0]] where
    T := Table
    heq := timesTableT_eq_Table
    hI := by
      simp only [pow_succ, pow_one, pow_zero, one_mul]
      exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0
    hmul := by decide

def E2RS0 : IdealEqCertificateO O ℤ timesTableO (Ideal.span {2} * I2N0) (Ideal.span {B.equivFun.symm ![0, 1, 0, 0, 0]} * J ^ 2) 
      ![![4, 0, 0, 0, 0], ![0, 0, 2, 0, 0]] ![![0, 2, 0, 0, 0], ![0, 2, 0, 2, 0]] where 
    T := Table
    heq := timesTableT_eq_Table
    hieq1 := by convert ideal_mul_span_singleton_coe O ℤ  _ _ _ rfl 2 ; decide ; exact {out := by decide}
    hieq2 := by 
      exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R2RS0
    g := ![![![1, -1, 1, 0, 0], ![3, 0, 0, 0, 1]], ![![-3, 3, 0, -2, 0], ![0, 2, 0, 3, 0]]]
    h := ![![![0, 0, 0, -1, 0], ![0, 1, 0, 0, 


def R17RS2 : IdealMulPrincipalCertificate O ℤ timesTableO (J ^ 2) ![10, 1, 0, 2, 2] ![![2, 0, 0, 0, 0], ![1, 0, 1, 0, 0]]
  ![![20, 2, 0, 4, 4], ![14, 0, 0, 2, 2]] where
    T := Table
    heq := timesTableT_eq_Table
    hI := by
      simp only [pow_succ, pow_one, pow_zero, one_mul]
      exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0
    hmul := by decide

def E17RS2 : IdealEqCertificateO O ℤ timesTableO (Ideal.span {2} * I17N2) (Ideal.span {B.equivFun.symm ![10, 1, 0, 2, 2]} * J ^ 2) 
      ![![34, 0, 0, 0, 0], ![-8, 2, 0, 0, 0]] ![![20, 2, 0, 4, 4], ![14, 0, 0, 2, 2]] where 
    T := Table
    heq := timesTableT_eq_Table
    hieq1 := by convert ideal_mul_span_singleton_coe O ℤ  _ _ _ rfl 2 ; decide ; exact {out := by decide}
    hieq2 := by 
      exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R17RS2
    g := ![![![-19, 19, 0, 2, 0], ![13, 0, 0, 13, 1]], ![![-8, -27, 49, 0, -12], ![20, 0, 0, -2, 24]]]
    h := ![![![9538, -4

In [666]:
#Write a term of type PrimesBelowBoundCertificte

#The ideals are names as follow: The i-th prime ideal above p is called I{p}N{i} in case there are more than one 
# prime ideals above p, otherwise the single ideal is I{p}N 

#The terms PrimesBelowPCertificate p that certify the prime factorization of (p) are called PBC{p}

#The proof of the norm of the i-th ideal above p is called N{p}N{i} (and N{p}N in case there is only a single ideal). 

#primes_below_proof is the proof of the primes below M 

def PrimesBelowBoundCertificteGen(K,B,M,primes_below_proof):
    #The list of lists of norms of ideals above p. 
    N = []
    I_names = []
    N_names = []
    for p in primes(M): 
        Lp = []
        I_names_p = []
        N_names_p = []
        for i in range(len(PrimesBelowGens(K,p))): 
            Lp = Lp + ((PrimesBelowGens(K,p))[i][1])*[(O.ideal((PrimesBelowGens(K,p))[i][0])).norm()]
            if len(PrimesBelowGens(K,p)) == 1 :
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N']
            else: 
                I_names_p = I_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'I{p}N{i}']
                N_names_p = N_names_p + ((PrimesBelowGens(K,p))[i][1]) * [f'NI{p}N{i}']
        N = N + [Lp]
        I_names = I_names + [I_names_p]
        N_names = N_names + [N_names_p]
                   
    m = len(N)
    #Nrm = [[N[i][j][0] for j in range(len(N[i]))] for i in range(m)]
    g = [len(N[i]) for i in range(m)]
        
    #Collects the indices of those ideals with norm below M 
    ideals_below = [[I_names[i][j] for j in range(g[i]) if N[i][j] < M] for i in range(m)]
    
    ideals_below.reverse()

    ideals_below_as_string = "[" + ", ".join("[" + ", ".join(sub) + "]" for sub in ideals_below) + "]"
    
    I_names.reverse()
    
    ideals_names_as_string = ["[" + ", ".join(sub) + "]" for sub in I_names]
    
    N_names.reverse()

    
    g.reverse()
    P = [p for p in primes(M)]
    P.reverse()
    N.reverse()
    
    print(f'''def PB{M} : PrimesBelowBoundCertificate O {M} where
  m := {m}
  g := !{g}
  P := !{P}
  hP := {primes_below_proof}
  I := fun i => by
    cases i
    rename_i i h
    interval_cases i ''')
    for i in range(m): 
        print(f'    · exact !{ideals_names_as_string[i]}')
    print(f'''  hC := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
    for i in range(m):
        print(f'    · exact PBC{P[i]}')
    print(f'''  N := fun i => by
    cases i
    rename_i i h
    interval_cases i''')
    for i in range(m):
        print(f'    · exact !{N[i]}')
    print(f'''  hN := fun i => by
     cases i
     rename_i i h
     interval_cases i ''')
    for i in range(m):
        print('''     · dsimp ; intro j
       fin_cases j''')
        for j in range(g[i]):
            print(f'       exact {N_names[i][j]}')
    print(f'  Il := !{ideals_below_as_string}')
    print('''  hIl := by
      intro i
      cases i
      rename_i i h
      interval_cases i
      all_goals rfl''')
    
    
        


In [667]:
PrimesBelowBoundCertificteGen(K,B,63,'primes_below_63')

def PB63 : PrimesBelowBoundCertificate O 63 where
  m := 18
  g := ![1, 1, 1, 1, 3, 3, 3, 1, 3, 3, 1, 3, 1, 3, 3, 5, 1, 5]
  P := ![61, 59, 53, 47, 43, 41, 37, 31, 29, 23, 19, 17, 13, 11, 7, 5, 3, 2]
  hP := primes_below_63
  I := fun i => by
    cases i
    rename_i i h
    interval_cases i 
    · exact ![I61N]
    · exact ![I59N]
    · exact ![I53N]
    · exact ![I47N]
    · exact ![I43N0, I43N1, I43N2]
    · exact ![I41N0, I41N1, I41N2]
    · exact ![I37N0, I37N1, I37N2]
    · exact ![I31N]
    · exact ![I29N0, I29N1, I29N2]
    · exact ![I23N0, I23N1, I23N2]
    · exact ![I19N]
    · exact ![I17N0, I17N1, I17N2]
    · exact ![I13N]
    · exact ![I11N0, I11N1, I11N2]
    · exact ![I7N0, I7N1, I7N2]
    · exact ![I5N, I5N, I5N, I5N, I5N]
    · exact ![I3N]
    · exact ![I2N0, I2N1, I2N1, I2N1, I2N1]
  hC := fun i => by
    cases i
    rename_i i h
    interval_cases i
    · exact PBC61
    · exact PBC59
    · exact PBC53
    · exact PBC47
    · exact PBC43
    · exact PBC41
    · exa

In [422]:
set_random_seed(10)

#Compute class group and let J be the generator. 

Cl = K.class_group('pari')
n = K.degree()
a = K.gen()
B = [K(x) for x in B]
ideal_gens = Cl.gens()[0].ideal().gens_reduced()
J = O.ideal(ideal_gens)

p = 7
F = PrimesBelowGens(K,p)
l = len(F)

#The ideals above p with norm less than B 

ideal_gens = [O.ideal(F[i][0]) for i in range(l) if O.ideal(F[i][0]).norm() < 63]
print(ideal_gens)


cont = 0
expon = (Cl(ideal_gens[cont])).exponents()[0]
genel = (ideal_gens[cont]/(J**expon)).gens_reduced()[0]
d = genel.denominator()

for p in prime_divisors(d):
        while (d % p == 0) and ((d//p) * genel in O):
            d = d // p

#d = 2
print('holaaaa',d)
x = d * genel
#print([x * i for i in (J ** 2).reduced_gens()])
A = [x * i for i in (J ** expon).gens_reduced()]
C = ([d * i for i in (ideal_gens[cont]).gens()])
CC = ideal_gens[cont].gens()

Gen1 = [elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))]
Gen2 = [elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))]
print([elems_to_basis(A, B).transpose().rows()[j].list() for j in range(len(A))])
print([elems_to_basis(C, B).transpose().rows()[j].list() for j in range(len(A))])

before_div = [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))]
print('before div =', [elems_to_basis(CC, B).transpose().rows()[j].list() for j in range(len(A))])
#print(elems_to_basis(C, B).transpose())
h = [IdealLift(K, B, C, A[i]) for i in range(len(A))]
g = [IdealLift(K, B, A, C[i]) for i in range(len(C))]

alpha = elems_to_basis([x], B).list()

print('alpha = ', alpha, 'denom =',d, 'exponent', expon)


gp = [[elems_to_basis(g[i], B).transpose().rows()[j].list() for j in range(len(A))] for i in range(len(g))]
hp = [[elems_to_basis(h[i], B).transpose().rows()[j].list() for j in range(len(C))] for i in range(len(h))]

print(ExList(str(gp)))
print(ExList(str(hp)))

numero = cont 

print(f'''
def R{p}RS{numero} : IdealMulPrincipalCertificate O ℤ timesTableO (J ^ {expon}) !{alpha} ![![2, 0, 0, 0, 0], ![1, 0, 1, 0, 0]]
  {ExList(str(Gen1))} where
T := Table
heq := timesTableT_eq_Table
hI := by
  simp only [pow_succ, pow_one, pow_zero, one_mul]
  exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0
hmul := by decide

def E{p}RS{numero} : IdealEqCertificateO O ℤ timesTableO (Ideal.span {d} * I{p}N{numero}) (Ideal.span B.equivFun.symm !{alpha} * J ^ {expon}) 
  {ExList(str(Gen2))} {ExList(str(Gen1))} where 
T := Table
heq := timesTableT_eq_Table
hieq1 := by 
  unfold I{p}N{numero}
  rw [Ideal.span_mul_span] ; congr
  simp only [Set.mem_singleton_iff, Set.mem_range, Set.iUnion_exists,
    Set.iUnion_iUnion_eq', Set.iUnion_singleton_eq_range, Set.iUnion_iUnion_eq_left, (show (2 : O) = ↑(2 : ℤ) by rfl), 
    ← zsmul_eq_mul _ 2, ← LinearEquiv.map_smul, B]
  have aux : (2 : ℤ) • {ExList(str(before_div))} = {ExList(str(Gen2))} := by decide
  refine congr_arg (fun x => Set.range x) ?_
  ext i ; congr  ; exact congr_fun aux i 
hieq2 := by 
  exact ideal_eq_principal_mul_of_IdealMulPrincipalCertificate O ℤ _ _ _ _ _ R{p}RS{numero}
g := {ExList(str(gp))}
h := {ExList(str(hp))}
hle1 := by decide
hle2 := by decide

lemma R{p}N{numero} : Ideal.span {d} * I{p}N{numero} =  Ideal.span B.equivFun.symm !{alpha} * J ^ {expon} := by 
  refine ideal_eq_of_IdealEqCertificateO _ _ _ _ _ _ _ E{p}RS{numero}




''')


[Fractional ideal (7, a - 3), Fractional ideal (7, a - 2)]
8
holaaaa 2
[[4, 2, 2, 4, 2], [6, -2, -2, 0, 2]]
[[14, 0, 0, 0, 0], [-6, 2, 0, 0, 0]]
before div = [[7, 0, 0, 0, 0], [-3, 1, 0, 0, 0]]
alpha =  [2, 1, 1, 2, 1] denom = 2 exponent 1
![![![-114, -36, -323, -36, -242], ![-91, 486, 162, 0, 0]], ![![38, 12, 108, 12, 81], ![31, -163, -54, 0, 0]]]
![![![278, -132, 201, -141, -2], ![648, 80, 517, 15, 0]], ![![273, -130, 193, -135, -2], ![636, 74, 495, 15, 0]]]

def R2RS0 : IdealMulPrincipalCertificate O ℤ timesTableO (J ^ 1) ![2, 1, 1, 2, 1] ![![2, 0, 0, 0, 0], ![1, 0, 1, 0, 0]]
  ![![4, 2, 2, 4, 2], ![6, -2, -2, 0, 2]] where
T := Table
heq := timesTableT_eq_Table
hI := by
  simp only [pow_succ, pow_one, pow_zero, one_mul]
  exact ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulJ0
hmul := by decide

def E2RS0 : IdealEqCertificateO O ℤ timesTableO (Ideal.span 2 * I2N0) (Ideal.span B.equivFun.symm ![2, 1, 1, 2, 1] * J ^ 1) 
  ![![14, 0, 0, 0, 0], ![-6, 2, 0, 0, 0]] ![!

In [220]:
#def IteratedMulLean (K, B, ideal_gens, ideal_pows, proof1_name, proof2_name, ideal_name):

IteratedMulLean(K, B, [[3,a], [3, -5/24*a^4 + 1/8*a^3 - 47/24*a^2 + 59/24*a + 13/4],[3,a + 1]], [1,1,1], 'rfl', 'rfl', 'I3N')

def MulI3N0 : IdealMulEqCertificate O ℤ timesTableO (I3N0) I3N1
  ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] ![![3, 0, 0, 0, 0], ![2, 3, -3, -1, -5]]
  ![![3, 0, 0, 0, 0], ![0, -6, 4, 1, 6]] where
 T := Table
 heq := timesTableT_eq_Table
 hI1 := rfl
 hI2 := rfl
 M :=  ![![![9, 0, 0, 0, 0], ![6, 9, -9, -3, -15]], ![![0, 3, 0, 0, 0], ![-3, 0, 1, 1, 3]]]
 hmul := by decide
 f :=  ![![![![-2, -1, 1, 0, 2], ![1, -3, 0, 0, 2]], ![![2, 1, 0, 0, 0], ![0, 0, 0, 0, 0]]], ![![![-7, 0, 0, -2, 1], ![0, -15, 0, 0, 10]], ![![8, 5, 0, 0, 0], ![4, 0, 0, 0, 0]]]]
 g :=  ![![![![3, 0, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![2, 3, -3, -1, -5], ![0, 0, 0, 0, 0]]], ![![![0, 1, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![-1, 2, -1, 0, -1], ![1, 0, 0, 0, 0]]]]
 hle1 := by decide
 hle2 := by decide

def MulI3N1 : IdealMulEqCertificate O ℤ timesTableO (I3N0*I3N1) I3N2
  ![![3, 0, 0, 0, 0], ![0, -6, 4, 1, 6]] ![![3, 0, 0, 0, 0], ![1, 1, 0, 0, 0]]
  ![![3, 0, 0, 0, 0]] where
 T := Table
 heq := timesTableT_eq_Table
 hI1 := ideal_eq_mul

(3,)

In [26]:
list_ideal_gens = [3,a]
ideal_name = 'I'

gensc = [elems_to_basis([x], B).list() for x in list_ideal_gens]
print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')

w = IdealEqSpanCertificateLean(K, list_ideal_gens, B, 'A', ideal_name)
ik, jk = FindNzEntry(w) 
print('')
print(f'lemma N : Nat.card (O ⧸ {ideal_name}{i}) = {O.ideal(list_ideal_gens).norm()} := ')
print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A)")            
print('')

def I : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] i)))
def A: IdealEqSpanCertificate O ℤ timesTableO I
 ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] 
 ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 3]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![3, 0, 0, 0, 0], ![0, 3, 0, 0, 0], ![0, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 3]], ![![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 1, 0, 2, 0], ![3, -1, 2, 3, 12], ![0, 0, 0, -2, -3]]]
  hmulB := by decide
  f := ![![![1, 0, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![0, 0, 0, 0, 0], ![1, 0, 0, 0, 0]], ![![0, 0, 0, 0, 0], ![0, 1, 0, 0, 0]], ![![0, -1, 0, -1, 0], ![1, 0, 2, 0, 0]], ![![0, 0, 0, 0, 1], ![0, 0, 0, 0, 0]]]
  g := ![![![1, 0, 0, 0, 0], ![0, 3, 0, 0, 0], ![0, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 1]], ![![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 1, 0, 2, 0], ![1, -1, 2, 3, 4], ![0, 0, 0, -2, -1]]]
  hle1 := by decide
 

In [139]:
set_random_seed(10)
g, f = RandomPrimitiveElementQuot(K, list_ideal_gens, 3)

In [142]:
f

x^2 + 2*x + 2

In [188]:
list((elems_to_basis([g],B)).transpose()[0])

[-17, 4, -15, -20, -68]

In [141]:
list((elems_to_basis([ZZ[X](f)(g)],B)).transpose())

[(5583, -3658, 3709, 4702, 22416)]

In [75]:
ZZ[X](f)(g) in I

False

In [143]:
IdealMemZLean(K, B,list_ideal_gens, [5583, -3658, 3709, 4702, 22416], 'MemC', 'I', 'A')

def MemC : IdealMemCertificate O ℤ B I
  ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 3]] ![5583, -3658, 3709, 4702, 22416] where
 hieq := ideal_eq_of_IdealEqSpanCertificate _ _ A
 g := ![1861, -3658, 3709, 4702, 7472]
 hmem := by decide


In [189]:
import sys
CertificateIrrModpFFP(GF(3)[X](f),3,0,sys.stdout)

def P3P0 : CertificateIrreducibleZModOfList' 3 2 2 1 [2, 2, 1] where
 m := 1
 P := ![2]
 exp := ![1] 
 hneq := by decide
 hP := by decide
 hlen := by decide
 htr := by decide
 bit := ![1, 1]
 hbits := by decide
 h := ![[0, 1], [1, 2], [0, 1]]
 g := ![![[1, 1]],![[2, 2]]]
 h' := ![![[1, 2], [0, 1]],![[0, 1], [1, 2]]]
 hs := by decide
 hz := by decide
 hmul := by decide
 a := ![[], [1]]
 b := ![[], [2, 2]]
 hhz := by decide
 hhn := by decide
 hgcd := by decide


In [65]:
len(B)

5